## Import packages

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import integrate
import os

import ADFWI
from ADFWI.model import AbstractModel, AcousticModel, AnisotropicElasticModel, IsotropicElasticModel
from ADFWI.propagator import AcousticPropagator, ElasticPropagator, GradProcessor, TorchGradProcessor
from ADFWI.survey import Receiver, SeismicData, Source, Survey
from ADFWI.utils import (
    build_anomaly_background_model,
    build_layer_model,
    calculate_spectrum,
    filter_low_frequencies_zero_phase,
    get_anomaly_model,
    get_linear_hess_model,
    get_linear_marmousi2_model,
    get_linear_vel_model,
    get_smooth_hess_model,
    get_smooth_layer_model,
    get_smooth_marmousi_model,
    get_smooth_valhall_model,
    load_hess_model,
    load_marmousi_model,
    load_overthrust_initial_model,
    load_overthrust_model,
    load_valhall_model,
    numpy2tensor,
    plot_filtered_data,
    plot_frequency_distribution,
    plot_spectrum,
    resample_marmousi_model,
    resample_overthrust_model,
    tensor2numpy,
    wavelet,
)
from ADFWI.utils.assessment_metric import MAPE, MSE, RMSE, SNR, SSIM
from ADFWI.utils.first_arrivel_picking import apply_mute, brutal_picker, mask, mute_arrival
from ADFWI.utils.noise import add_gaussian_noise
from ADFWI.utils.offset_mute import mute_offset
from ADFWI.view import (
    animate_inversion_process,
    plot_bcx_bcz,
    plot_damp,
    plot_eps_delta_gamma,
    plot_initial_and_inverted,
    plot_lam_mu,
    plot_misfit,
    plot_model,
    plot_survey,
    plot_vp_rho,
    plot_vp_vs_rho,
    plot_waveform2D,
    plot_waveform_trace,
    plot_waveform_wiggle,
    plot_wavelet,
)
from ADFWI.fwi import AcousticFWI, ElasticFWI
from ADFWI.fwi.misfit import (
    Misfit,
    Misfit_NIM,
    Misfit_envelope,
    Misfit_global_correlation,
    Misfit_sdtw,
    Misfit_traveltime,
    Misfit_wasserstein_sinkhorn,
    Misfit_waveform_L1,
    Misfit_waveform_L2,
    Misfit_waveform_SquaredL2,
    Misfit_waveform_smoothL1,
    Misfit_waveform_studentT,
    Misfit_weighted_DTW_GC,
    Misfit_weighted_ECI,
    Misfit_weighted_L1_and_L2,
)
from ADFWI.fwi.regularization import (
    Regularization,
    regularization_TV_1order,
    regularization_TV_2order,
    regularization_Tikhonov_1order,
    regularization_Tikhonov_2order,
)

project_path = "./data"
os.makedirs(os.path.join(project_path,"model"), exist_ok=True)
os.makedirs(os.path.join(project_path,"waveform"), exist_ok=True)
os.makedirs(os.path.join(project_path,"survey"), exist_ok=True)
os.makedirs(os.path.join(project_path,"inversion"), exist_ok=True)


## Define the basic model parameter

In [ ]:
device = "npu:0"         # Specify the CPU/GPU/NPU device
dtype = torch.float32     # Set data type to 32-bit floating point
backend = ADFWI.set_backend(device, dtype=dtype)
ox,oz  = 0,0
nz,nx  = 78,200
dx,dz  = 45, 45
nt,dt  = 2500, 0.003
nabc   = 50
f0     = 3
free_surface = True

## Define the initial velocity model

In [ ]:
# Load the Marmousi model dataset from the specified directory.
marmousi_model = load_marmousi_model(in_dir="../../datasets/marmousi2_source")
x         = np.linspace(5000, 5000+dx*nx, nx)
z         = np.linspace(0, dz*nz, nz)
vel_model = resample_marmousi_model(x, z, marmousi_model)
vp_true   = vel_model['vp'].T
vs_true   = vel_model['vs'].T
rho_true = np.ones_like(vp_true)*2450

smooth_model= get_smooth_marmousi_model(vel_model,gaussian_kernel=4,mask_extra_detph=2,rcv_depth=8)
vp_init     = smooth_model['vp'].T
vs_init     = smooth_model['vs'].T
rho_init = np.ones_like(vp_init)*2450
vp_init[:10]= vp_true[:10]
vs_init[:10]= vs_init[:10]
rho_init[:10]=rho_true[:10] 

water_layer_mask = np.zeros_like(vp_init)
water_layer_mask[:10] = 1

# processing the water layer
model = IsotropicElasticModel(
                ox,oz,nx,nz,dx,dz,
                vp_init,vs_init,rho_init,
                vp_bound =[vp_true.min(),vp_true.max()],
                vs_bound =[vs_true.min(),vs_true.max()],
                # rho_bound=[rho_true.min(),rho_true.max()],
                vp_grad = True,vs_grad = True, rho_grad=False,
                auto_update_rho=False,auto_update_vp=False,
                free_surface=free_surface,
                abc_type="PML",abc_jerjan_alpha=0.007,nabc=nabc,
                water_layer_mask=water_layer_mask)


model.save(os.path.join(project_path,"model/init_model.npz"))
print(model.__repr__())

In [ ]:
model._plot_vp_vs_rho(figsize=(12,5),wspace=0.2,cbar_pad_fraction=0.18,cbar_height=0.04,cmap='coolwarm',save_path=os.path.join(project_path,"model/init_vp_vs_rho.png"),show=True)

## Define the observed systems: Survey = Source + Receiver

In [ ]:
# source    
src_z = np.array([2    for i in range(2,nx-2,5)]) 
src_x = np.array([i    for i in range(2,nx-2,5)])
src_t,src_v = wavelet(nt,dt,f0,amp0=1)
src_v = integrate.cumtrapz(src_v, axis=-1, initial=0) #Integrate
source = Source(nt=nt,dt=dt,f0=f0)
# Method 2: Loop through each source position to add them individually
for i in range(len(src_x)):
    source.add_source(src_x=src_x[i], src_z=src_z[i], src_wavelet=src_v, src_type="mt", src_mt=np.array([[1,0,0],[0,1,0],[0,0,1]]))

In [ ]:
# receiver
rcv_z = np.array([2   for i in range(0,nx,1)])
rcv_x = np.array([j   for j in range(0,nx,1)])
receiver = Receiver(nt=nt,dt=dt)
for i in range(len(rcv_x)):
    receiver.add_receiver(rcv_x=rcv_x[i], rcv_z=rcv_z[i], rcv_type="pr")
# survey
survey = Survey(source=source,receiver=receiver)

In [ ]:
print(survey.__repr__())
survey.plot(model.vp,cmap='coolwarm',save_path=os.path.join(project_path,"survey/observed_system_init.png"),show=True)

In [ ]:
# Plot the wavelet used in the source
source.plot_wavelet(save_path=os.path.join(project_path,"survey/wavelets.png"),show=True)

In [ ]:
# Plot the survey configuration over the velocity model
survey.plot(model.vp,cmap='coolwarm',save_path=os.path.join(project_path,"survey/observed_system_init.png"))

## Inversion

In [ ]:
from ADFWI.fwi.misfit import Misfit_waveform_L2
from ADFWI.fwi.regularization import regularization_TV_2order

# Setup misfit function
loss_fn = Misfit_waveform_L2(dt=dt)
regularization_fn = regularization_TV_2order(nx,nz,dx,dz,step_size=50,gamma=1)

# gradient processor
grad_mask = np.ones_like(vp_init)
grad_mask[:10] = 0
gradient_processor = GradProcessor(grad_mask=grad_mask,forw_illumination=False)

# Initialize the wave propagator using the specified model and survey configuration
F = ElasticPropagator(model,survey)

# load data
d_obs = SeismicData(survey)
d_obs.load(os.path.join(project_path,"waveform/obs_data.npz"))

iterations  = [1,1,1]
# iterations  = [100,100,100]
freqs       = [2, 3, 5]
lrs         = [10, 6, 2]

start_iter = 0
iter_vp,iter_vs,iter_rho,iter_loss = [],[],[],[]
for iteration,freq,lr in zip(iterations,freqs,lrs):
    # optimizer
    optimizer   =   torch.optim.Adam(model.parameters(), lr = 10)
    scheduler   =   torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.75, last_epoch=-1)

    # Initialize the acoustic full waveform inversion (FWI) object.
    fwi = ElasticFWI(propagator=F,
                     model=model,
                     optimizer=optimizer,
                     scheduler=scheduler,
                     loss_fn=loss_fn,
                     regularization_fn=regularization_fn,
                     regularization_weights_x=[1e-6,1e-5,0,0,0,0],
                     regularization_weights_z=[1e-6,1e-5,0,0,0,0],
                     obs_data=d_obs,
                     gradient_processor=gradient_processor,
                     waveform_normalize=True,
                     cache_result=True,
                     save_fig_epoch=1,
                     save_fig_path=os.path.join(project_path,"inversion"),
                     inversion_component=["vx","vz"]
                     )
    
    # Run the forward modeling for the specified number of iterations.
    fwi.forward(iteration=iteration,fd_order=4,
                        batch_size=None,checkpoint_segments=4,
                        start_iter=start_iter,
                        cutoff_freq=freq
                        )
    start_iter += iteration
    
    # Retrieve the inversion results: updated velocity and loss  values.
    iter_vp.extend(fwi.iter_vp)
    iter_vs.extend(fwi.iter_vs)
    iter_rho.extend(fwi.iter_rho)
    iter_loss.extend(fwi.iter_loss)

# Save the iteration results to files for later analysis.
np.savez(os.path.join(project_path,"inversion/iter_vp.npz"),data=np.array(iter_vp))
np.savez(os.path.join(project_path,"inversion/iter_vs.npz"),data=np.array(iter_vs))
np.savez(os.path.join(project_path,"inversion/iter_rho.npz"),data=np.array(iter_rho))
np.savez(os.path.join(project_path,"inversion/iter_loss.npz"),data=np.array(iter_loss))

## visualize the inverted results

In [ ]:
# plot the misfit
plt.figure(figsize=(8,6))
plt.plot(iter_loss,c='k')
plt.xlabel("Iterations", fontsize=12)
plt.ylabel("L2-norm Misfits", fontsize=12)
plt.tick_params(labelsize=12)
plt.savefig(os.path.join(project_path,"inversion/misfit.png"),bbox_inches='tight',dpi=100)
plt.show()

In [ ]:
# plot the initial model and inverted resutls
plt.figure(figsize=(12,8))
plt.subplot(121)
plt.imshow(vp_init,cmap='jet_r')
plt.colorbar(orientation='horizontal')
plt.subplot(122)
plt.imshow(iter_vp[-1],cmap='jet_r')
plt.colorbar(orientation='horizontal')
plt.savefig(os.path.join(project_path,"inversion/inverted_vp.png"),bbox_inches='tight',dpi=100)
plt.show()

In [ ]:
# plot the initial model and inverted resutls
plt.figure(figsize=(12,8))
plt.subplot(121)
plt.imshow(vs_init,cmap='jet_r')
plt.colorbar(orientation='horizontal')
plt.subplot(122)
plt.imshow(iter_vs[-1],cmap='jet_r')
plt.colorbar(orientation='horizontal')
plt.savefig(os.path.join(project_path,"inversion/inverted_vs.png"),bbox_inches='tight',dpi=100)
plt.show()

In [ ]:
# plot the initial model and inverted resutls
plt.figure(figsize=(12,8))
plt.subplot(121)
plt.imshow(rho_init,cmap='jet_r')
plt.colorbar(orientation='horizontal')
plt.subplot(122)
plt.imshow(iter_rho[-1],cmap='jet_r')
plt.colorbar(orientation='horizontal')
plt.savefig(os.path.join(project_path,"inversion/inverted_rho.png"),bbox_inches='tight',dpi=100)
plt.show()

In [ ]:
# plot the gradient
plt.figure()
plt.subplot(1,2,1)
plt.imshow(fwi.iter_vp_grad[-1],cmap='coolwarm')
plt.colorbar(orientation='horizontal')
plt.subplot(1,2,2)
plt.imshow(fwi.iter_vs_grad[-1],cmap='coolwarm')
plt.colorbar(orientation='horizontal')
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
plt.imshow(fwi.iter_vp_grad[-1],cmap='seismic',vmin=-4000,vmax=4000)
plt.colorbar(orientation='horizontal')
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
plt.imshow(fwi.iter_vs_grad[-1],cmap='seismic',vmin=-800,vmax=800)
plt.colorbar(orientation='horizontal')
plt.show()